# RecSys RL Gym — Meal Agent (Colab)

Train the meal-planning agent on a **CUDA GPU**: real Food.com data → **SFT** warm-start → **GRPO** (RLVR) → comparison table.

**First:** Runtime ▸ Change runtime type ▸ **GPU**. Free **T4** works with the 0.5B model; **A100/L4** (Colab Pro) for 1.5B and faster.

In [ ]:
import torch
print('CUDA :', torch.cuda.is_available())
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime to GPU!')
print('bf16 :', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## 1. Get the code
Push the repo to GitHub and set `REPO_URL`, **or** comment it out and upload a zip with the fallback cell.

In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/<you>/rl-gym.git'
if REPO_URL:
    !git clone $REPO_URL rlgym
    %cd rlgym
else:
    print('No REPO_URL — use the upload fallback below, then %cd into the folder.')

In [ ]:
# Get the code onto this machine. Works on Colab AND plain Jupyter / SSH GPU boxes.
# If rl_gym/ is already here (you cloned or rsync'd the repo), this is a no-op.
import os, glob, zipfile

if not os.path.isdir('rl_gym'):
    try:
        from google.colab import files          # Colab only
        up = files.upload()                      # pick rl-gym-code.zip
        zip_path = list(up)[0]
    except ModuleNotFoundError:
        # Not Colab. Upload rl-gym-code.zip via the Jupyter file browser (Upload
        # button) or scp/rsync it next to this notebook, then re-run this cell.
        zips = sorted(glob.glob('*.zip'))
        assert zips, (
            f'No rl_gym/ and no .zip in {os.getcwd()}. Upload rl-gym-code.zip '
            'via the Jupyter file browser (or scp it here), then re-run this cell.'
        )
        zip_path = zips[0]
        print('extracting', zip_path)
    zipfile.ZipFile(zip_path).extractall('.')   # zip has no wrapper folder -> cwd is repo root

assert os.path.isdir('rl_gym'), \
    f'rl_gym/ not found in {os.getcwd()} — cwd has: {os.listdir(".")}'
print('code ready in', os.getcwd())


## 2. Install deps
Pinned to the validated stack (works on Colab's torch). vLLM is optional — a big speed-up on A100/L4, but left out by default for reliability.

In [ ]:
!pip -q install 'trl>=0.14,<0.17' 'transformers>=4.48,<5' 'datasets<4' accelerate huggingface_hub
!pip -q install vllm   # then remove --no_vllm in the GRPO cell
import trl, transformers; print('trl', trl.__version__, '| transformers', transformers.__version__)

## 3. Real data (Food.com via HF — no Kaggle/token needed)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download
os.makedirs('data/foodcom', exist_ok=True)
p = hf_hub_download('AkashPS11/recipes_data_food.com', 'recipes.csv', repo_type='dataset')
shutil.copy(p, 'data/foodcom/recipes.csv')
print('data ready:', os.path.getsize('data/foodcom/recipes.csv') // 10**6, 'MB')

## 4. Config
Use **1.5B** on A100/L4; switch to **0.5B** on a free T4.

In [ ]:
import os
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # free T4: 'Qwen/Qwen2.5-0.5B-Instruct'
DATA  = './data/foodcom'
assert os.path.isdir('rl_gym'), (
    f'rl_gym/ not in cwd ({os.getcwd()}) — get the code first (cell 3 clone '
    f'or cell 4 upload) and run from the repo root. cwd has: {os.listdir(".")}'
)
print('OK — repo root, model =', MODEL)


## 5. SFT warm-start — teach valid PLAN format on real data
Base model can't emit a valid plan zero-shot; this fixes that (and gives a strong baseline).

In [ ]:
!python -m rl_gym.meals.sft --model {MODEL} --data_dir {DATA} --n_episodes 1200 --epochs 2 --output_dir ./out/meals-sft

## 6. GRPO (RLVR) from the SFT checkpoint
Closes the gap to oracle. If you installed vLLM, **remove `--no_vllm`** for a big speed-up. If loss/grad ever explode, add `--no_scale_rewards --beta 0.0`.

In [ ]:
!python -m rl_gym.gym.train --model ./out/meals-sft --data_dir {DATA} --num_generations 8 --max_steps 150 --lr 1e-6 --beta 0.04 --max_completion_len 32 --max_prompt_len 1024 --no_vllm --output_dir ./out/meals-grpo

## 7. Comparison table — the pitch artifact
random → base → SFT → GRPO → oracle, on real dev data.

In [ ]:
!python -m rl_gym.meals.compare --data_dir {DATA} --base {MODEL} --sft ./out/meals-sft --grpo ./out/meals-grpo --n_episodes 300

## 8. (optional) Save checkpoints to Drive (survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rlgym_out && cp -r ./out/* /content/drive/MyDrive/rlgym_out/
print('saved to Drive/rlgym_out')